# ISAN 3392: Texas Wildlife Camera

Run Stage 1 before training. This notebook uses the same small functions as the app. No wildlife data is supplied. Select the project `.venv` kernel. See the README for exact Windows setup commands.

In [ ]:
from pathlib import Path
import os
import sys

# Work from the project root whether Jupyter started here or in notebooks/.
if not Path('wildlife').is_dir() and Path('../wildlife').is_dir():
    os.chdir('..')
assert Path('wildlife').is_dir(), 'Open this notebook from the project folder.'
print('Python:', sys.executable)


## 1. Check the environment

CUDA availability and an actual GPU operation are checked. The installed dependency list is saved for reproducibility.

In [ ]:
from check_environment import main as check_environment
check_environment()

## 2. Predict on your own image

Replace the filename with a real image. YOLO11n has a pretrained cow class, but no deer or wild boar class. Its CNN extracts visual features; the detection head predicts boxes and classes. This cell runs inference, not training.

In [ ]:
from wildlife.detection import load_model, detect_frame
import cv2
import matplotlib.pyplot as plt

image_path = Path('data/cow.jpg')
if image_path.is_file():
    model = load_model('yolo11n.pt')  # Load weights once, outside frame processing.
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError('The image could not be decoded.')
    annotated, animals = detect_frame(model, image, confidence=0.5)
    plt.figure(figsize=(9, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()
    print(animals)
else:
    print('Add your own real image at', image_path, 'then run this cell again.')

## 3. Process a saved video

This calls the standalone OpenCV demo. It samples at 3 FPS, writes a REPLAY video, and avoids retaining all frames in memory. A saved screen recording is also valid.

In [ ]:
import subprocess

video_path = Path('data/trail_recording.mp4')
if video_path.is_file():
    subprocess.run([sys.executable, 'detect.py', str(video_path), '--fps', '3'], check=True)
else:
    print('Add your own saved video at', video_path)


## 4. Understand alert confirmation

The numbers below are an illustrative Python rule example, not wildlife data or model measurements. Three consecutive positive frames trigger an event. Cooldown reduces duplicate alerts.

In [ ]:
from wildlife.events import AlertGate

gate = AlertGate(repeats=3, cooldown=30)
for seconds in [0.0, 0.34, 0.68, 1.02]:
    print(seconds, gate.update({'cow': 0.90}, seconds))


## 5. Supervised transfer learning and honest evaluation

Humans must annotate class IDs and normalized bounding boxes. Keep related frames, bursts, and source videos in one split. Training updates model weights using labeled examples; validation helps choose settings. The untouched test set estimates performance on independent examples. Precision measures correctness of predictions, recall measures coverage of labeled animals, and mAP summarizes class and localization performance. Confidence filtering and alert timing are Python rules, not learned layers.

Prepare the downloaded annotations with `prepare_dataset.py`, review the contact sheets and source groups, then use `data/wildlife_dataset/dataset.yaml`. The exact classes are `0: deer`, `1: wild_boar`, `2: cow`. When ready to train later, run these commands in PowerShell with the app stopped:

```powershell
.\.venv\Scripts\python.exe train.py --data data/wildlife_dataset/dataset.yaml --imgsz 640 --batch 4 --epochs 20
.\.venv\Scripts\python.exe evaluate.py --data data/wildlife_dataset/dataset.yaml --model runs/train/wildlife/weights/best.pt
```

Read the generated `per_class_metrics.csv`. Do not fill a report with invented scores. Replace the run folder if training printed `wildlife2`, etc. Enter the resulting `best.pt` path in the app.

## Local web dashboard

Start the separate FastAPI server from PowerShell:

```powershell
.\.venv\Scripts\python.exe -m uvicorn app:app --host 127.0.0.1 --port 8000
```

Open http://127.0.0.1:8000. Debug samples browser webcam/screen frames or replays an upload; Demo uses independent prerecorded cameras. JavaScript only captures and displays images. Python runs the CNN and the same alert gate demonstrated above. Keep training separate from the server.


## 6. Course reflection

Record: dataset source and permission; number of independent groups per split; class counts; exact environment and commands; precision, recall and AP for each class; examples of false positives and missed animals. Explain how low light, distance, class imbalance, and sampling at 3 FPS can affect results.

What did pretraining provide? What changed during fine-tuning? Why can nearby video frames in different splits inflate scores? How does changing confidence affect precision and recall?